# Notebook 02: Adversarial Examples — FGSM

## Why This Matters
In 2013, a paper showed that you could take an image correctly classified as a school bus, add a small noise pattern invisible to any human, and the classifier would confidently call it an ostrich. The noise wasn't random — it was precisely engineered from the model's own gradients. The discovery changed how the field thought about neural network reliability and kicked off a decade of adversarial ML research.

FGSM (Fast Gradient Sign Method) is the simplest, fastest, and most important adversarial attack to understand. Everything in adversarial ML builds from it — PGD, C&W, AutoAttack, physical-world attacks, certified defenses. If you understand FGSM at the gradient level, the rest of the literature opens up.

## Structure
1. What is an adversarial example?
2. Why neural networks are vulnerable — the geometry of high-dimensional loss landscapes
3. FGSM: the algorithm, from intuition to code
4. Targeted vs untargeted attacks
5. The epsilon-accuracy tradeoff
6. Transferability: why adversarial examples cross model boundaries
7. Defenses — what works, what doesn't, and why
8. Practical implications for deployed systems

## What You'll Learn
- Why the sign of the gradient — not its magnitude — is what matters for FGSM
- How a single gradient step produces a misclassification
- Why adversarial examples transfer across architectures (and what that implies about defense)
- Which "defenses" are security theater and which are real
- What ε means and how to reason about the perceptibility-robustness tradeoff

## Background

### The discovery

In 2013, Szegedy et al. at Google published *"Intriguing Properties of Neural Networks"* — a paper that set out to understand what deep networks had actually learned. One experiment stood out: they found that by solving a constrained optimization problem, they could construct inputs that were nearly identical to real images but caused confident misclassification. Importantly, the perturbations were too small for humans to reliably detect. The same network that achieved near-human accuracy on ImageNet would mistake a slightly-perturbed panda image for a gibbon.

This was disturbing for two reasons. First, it suggested that the internal representations learned by neural networks were less "robust" than human perception — the network was using features that humans don't, and those features could be exploited. Second, the perturbations *transferred*: an adversarial example crafted against one network often fooled other networks trained on the same data with different architectures. This meant the vulnerability was a property of the data distribution and the loss surface, not just a quirk of a specific model.

### Goodfellow's insight: it's the linearity

A year later, Goodfellow, Shlens & Szegedy (2014) published *"Explaining and Harnessing Adversarial Examples"* — one of the most-cited papers in deep learning. They proposed a much simpler explanation than Szegedy's optimization approach, and introduced FGSM.

The key insight: neural networks are roughly **linear in their inputs** in a local neighborhood (especially in high-dimensional spaces). And linearity in high dimensions is exploitable in a way that isn't obvious in 2D.

Consider a linear model $f(x) = w^T x$. If you add a small perturbation $\eta$ to $x$, the change in output is $w^T \eta$. Now suppose $x$ is 1000-dimensional and $\eta$ is constrained to have $\|\eta\|_\infty \leq \epsilon$ (each coordinate perturbed by at most $\epsilon$). The maximum change in output you can achieve is $\epsilon \cdot \|w\|_1$. If $w$ has 1000 components each of magnitude 0.5, that's $\epsilon \cdot 500$ — potentially very large even for small $\epsilon$.

The perturbation that maximizes this is $\eta = \epsilon \cdot \text{sign}(w)$ — take the sign of each weight component. Hence: **Fast Gradient Sign Method**.

For a neural network, $w$ is replaced by the gradient of the loss with respect to the input, $\nabla_x \mathcal{L}(f(x), y)$. Taking the sign of this gradient and scaling by $\epsilon$ gives the perturbation that most efficiently increases the loss — moving the input toward the decision boundary and across it.

### Why this matters beyond the panda image

The adversarial example demo (panda → gibbon) is famous but can feel like a toy. The practical implications are real:

**Autonomous vehicles.** Eykholt et al. (2018) showed that physical stickers on stop signs could cause vision systems to misclassify them as speed limit signs at high confidence. The stickers survived printing, mounting, and camera capture.

**Face recognition.** Sharif et al. (2016) showed that specially printed glasses could cause face recognition systems to misidentify the wearer as a specific target person. The glasses are wearable and the attack survives real-world camera capture.

**Medical imaging.** Finlayson et al. (2019) showed that adversarial attacks against medical AI systems could cause both under- and over-diagnosis — with implications for insurance fraud and patient harm.

**Content moderation.** Adversarial perturbations can cause content classifiers to pass harmful content as safe, or flag benign content as harmful.

### The arms race that followed

FGSM was fast and cheap but not particularly strong — a single gradient step often doesn't find the minimum-distortion adversarial example. This led to more powerful attacks:
- **PGD** (Madry et al., 2018): iterated FGSM with projection — currently the standard benchmark attack (notebook 03)
- **C&W** (Carlini & Wagner, 2017): optimization-based attack minimizing distortion directly — finds more efficient adversarial examples than PGD
- **AutoAttack** (Croce & Hein, 2020): ensemble of attacks for reliable evaluation

On the defense side: adversarial training (training on adversarial examples) is still the most reliable defense. Most other proposed defenses — input preprocessing, detection-based approaches, obfuscated gradients — have been broken. Athalye et al. (2018) broke 7 of 9 ICLR 2018 defense papers within months of publication.

In [ ]:
%pip install -q torch torchvision matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# Device selection: MPS on Apple Silicon, CUDA, then CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Setup: A Small CNN on MNIST

We'll work with MNIST — simple enough that the full attack loop runs in seconds, complex enough to illustrate everything we need. The same code applies to any differentiable model.

We train a small CNN to >98% accuracy, then attack it. This is realistic — FGSM is most instructive against a well-trained model, not a broken one.

In [ ]:
# Data loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean/std
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=256, shuffle=False)

print(f"Train: {len(train_dataset):,} images")
print(f"Test:  {len(test_dataset):,} images")
print(f"Input shape: {train_dataset[0][0].shape}")

In [ ]:
class MnistCNN(nn.Module):
    """Small CNN — 2 conv layers + 2 fc layers. ~99% accuracy on MNIST."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.drop  = nn.Dropout(0.25)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))   # (B, 32, 28, 28)
        x = self.pool(x)             # (B, 32, 14, 14)
        x = F.relu(self.conv2(x))   # (B, 64, 14, 14)
        x = self.pool(x)             # (B, 64, 7, 7)
        x = x.flatten(1)             # (B, 64*7*7)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)           # (B, 10) — logits


def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total


model = MnistCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training...")
for epoch in range(5):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        optimizer.step()
    acc = evaluate(model, test_loader, device)
    print(f"  Epoch {epoch+1}/5 — test acc: {acc:.4f}")

print(f"\nFinal clean accuracy: {evaluate(model, test_loader, device):.4f}")

## 2. FGSM: The Algorithm

Given:
- A model $f$ with parameters $\theta$
- A loss function $\mathcal{L}$ (cross-entropy)
- An input $x$ with true label $y$
- A perturbation budget $\epsilon$

FGSM produces an adversarial example $x^*$:

$$x^* = x + \epsilon \cdot \text{sign}(\nabla_x \mathcal{L}(f_\theta(x), y))$$

That's it. One forward pass to compute the loss, one backward pass to get the gradient, take the sign, scale by $\epsilon$.

**Why the sign?** We want to move $x$ in the direction that *most increases the loss*, subject to an $L_\infty$ constraint (each pixel perturbed by at most $\epsilon$). Under $L_\infty$, the optimal perturbation at each coordinate is to move it by exactly $\epsilon$ in the direction of the gradient — which is $\epsilon \cdot \text{sign}(\nabla_x \mathcal{L})$.

**Why $L_\infty$?** It's the most natural constraint for image imperceptibility — you want no individual pixel to be obviously different. Other norms ($L_2$, $L_0$) are also used in the literature but $L_\infty$ is the standard for FGSM.

In [ ]:
def fgsm_attack(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    """
    Untargeted FGSM attack.

    Args:
        model: the target model (must support gradients)
        x: clean input batch, shape (B, C, H, W)
        y: true labels, shape (B,)
        epsilon: L-inf perturbation budget

    Returns:
        x_adv: adversarial examples, same shape as x
    """
    x_adv = x.clone().detach().requires_grad_(True)

    # Forward pass: compute loss at clean input
    logits = model(x_adv)
    loss = F.cross_entropy(logits, y)

    # Backward pass: gradient of loss w.r.t. input
    model.zero_grad()
    loss.backward()

    # The adversarial perturbation: move in direction that increases loss
    grad_sign = x_adv.grad.sign()  # sign of gradient, shape (B, C, H, W)

    # Add perturbation, clamp to valid pixel range
    x_adv = x_adv.detach() + epsilon * grad_sign

    # Keep perturbation within the epsilon ball AND within valid input range
    # For normalized MNIST: roughly [-2.8, 2.8]
    x_adv = torch.clamp(x_adv, x.min().item(), x.max().item())

    return x_adv


# Walkthrough with a single example
model.eval()
x_batch, y_batch = next(iter(test_loader))
x_batch, y_batch = x_batch.to(device), y_batch.to(device)

# Pick one image
x_single = x_batch[:1]
y_single = y_batch[:1]

x_adv_single = fgsm_attack(model, x_single, y_single, epsilon=0.3)

# Check what the model predicts
with torch.no_grad():
    logits_clean = model(x_single)
    logits_adv   = model(x_adv_single)

pred_clean = logits_clean.argmax(dim=1).item()
pred_adv   = logits_adv.argmax(dim=1).item()
conf_clean = F.softmax(logits_clean, dim=1).max().item()
conf_adv   = F.softmax(logits_adv,   dim=1).max().item()

print(f"True label:              {y_single.item()}")
print(f"Clean prediction:        {pred_clean}  (confidence: {conf_clean:.3f})")
print(f"Adversarial prediction:  {pred_adv}  (confidence: {conf_adv:.3f})")
print(f"Attack successful:       {pred_adv != y_single.item()}")
print()
print(f"Perturbation L-inf norm: {(x_adv_single - x_single).abs().max().item():.4f}  (epsilon={0.3})")
print(f"Perturbation L-2 norm:   {(x_adv_single - x_single).norm().item():.4f}")

In [ ]:
# Visualize: clean image, perturbation, adversarial image
def unnormalize(x):
    """Undo MNIST normalization for visualization."""
    return x * 0.3081 + 0.1307

x_vis   = unnormalize(x_single.cpu()).squeeze().numpy()
x_adv_vis = unnormalize(x_adv_single.cpu()).squeeze().numpy()
perturbation = (x_adv_single - x_single).cpu().squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))

axes[0].imshow(x_vis, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Clean\npredicted: {pred_clean} ({conf_clean:.2f})', fontweight='bold')
axes[0].axis('off')

# Perturbation: centered colormap so 0 = gray
im = axes[1].imshow(perturbation, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[1].set_title(f'Perturbation × 1\n(ε={0.3}, L∞ bounded)', fontweight='bold')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

axes[2].imshow(np.clip(x_adv_vis, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Adversarial\npredicted: {pred_adv} ({conf_adv:.2f})', fontweight='bold',
                  color='red' if pred_adv != y_single.item() else 'black')
axes[2].axis('off')

fig.suptitle("FGSM: Clean → Perturbation → Adversarial", fontsize=12)
plt.tight_layout()
plt.savefig("fgsm_example.png", bbox_inches='tight')
plt.show()

## 3. The Epsilon–Accuracy Tradeoff

The perturbation budget $\epsilon$ controls a fundamental tradeoff:
- **Small $\epsilon$:** perturbation is imperceptible, but may not cross the decision boundary
- **Large $\epsilon$:** attack almost always succeeds, but perturbation becomes visible

This is the core tension in adversarial ML — you can't simultaneously have imperceptible perturbations and guaranteed misclassification. All defenses that claim to achieve both are suspect.

In [ ]:
def evaluate_under_fgsm(model, loader, epsilon, device, n_batches=20):
    """Evaluate model accuracy under FGSM attack across multiple batches."""
    model.eval()
    correct = total = 0
    for i, (x, y) in enumerate(loader):
        if i >= n_batches:
            break
        x, y = x.to(device), y.to(device)
        x_adv = fgsm_attack(model, x, y, epsilon)
        with torch.no_grad():
            preds = model(x_adv).argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / total


epsilons = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
print("Evaluating accuracy under FGSM at various epsilon values...")
accuracies = []
for eps in epsilons:
    acc = evaluate_under_fgsm(model, test_loader, eps, device)
    accuracies.append(acc)
    print(f"  ε={eps:.2f}  →  accuracy: {acc:.4f}  (attack success rate: {1-acc:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: accuracy vs epsilon
axes[0].plot(epsilons, accuracies, 'o-', color='steelblue', linewidth=2, markersize=6)
axes[0].axhline(0.1, color='red', linestyle='--', alpha=0.5, label='Random (10-class)')
axes[0].set_xlabel('Epsilon (L∞ budget)')
axes[0].set_ylabel('Model Accuracy')
axes[0].set_title('Clean CNN Accuracy under FGSM')
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].annotate(f'Clean: {accuracies[0]:.3f}', (0, accuracies[0]),
                 textcoords='offset points', xytext=(10, -15), fontsize=9)

# Right: examples at different epsilon
sample_x = x_batch[:1]
sample_y = y_batch[:1]
eps_viz = [0.0, 0.1, 0.2, 0.3, 0.5]

for i, eps in enumerate(eps_viz):
    if eps == 0:
        x_show = sample_x
    else:
        x_show = fgsm_attack(model, sample_x, sample_y, eps)
    with torch.no_grad():
        pred = model(x_show).argmax(dim=1).item()
        conf = F.softmax(model(x_show), dim=1).max().item()

    ax = axes[1]
    # Inset axes for each epsilon
    inset = ax.inset_axes([i/len(eps_viz), 0.1, 0.9/len(eps_viz), 0.8])
    inset.imshow(np.clip(unnormalize(x_show.cpu()).squeeze().numpy(), 0, 1), cmap='gray')
    color = 'red' if pred != sample_y.item() else 'black'
    inset.set_title(f'ε={eps}\n→{pred} ({conf:.2f})', fontsize=7, color=color)
    inset.axis('off')

axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].axis('off')
axes[1].set_title('Visual Effect of Epsilon (red = misclassified)')

plt.tight_layout()
plt.savefig("epsilon_tradeoff.png", bbox_inches='tight')
plt.show()

## 4. Targeted FGSM

Untargeted FGSM just wants the model to predict *anything wrong*. Targeted FGSM wants the model to predict a *specific* wrong class — more powerful, more useful for real attacks.

The change is simple: instead of maximizing the loss for the true label, we **minimize the loss for the target label**:

$$x^* = x - \epsilon \cdot \text{sign}(\nabla_x \mathcal{L}(f_\theta(x), y_{\text{target}}))$$

Note the **minus sign** — we're descending the loss for the target class, making the model more confident in it.

In [ ]:
def fgsm_targeted(
    model: nn.Module,
    x: torch.Tensor,
    y_target: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    """
    Targeted FGSM: push model toward predicting y_target.
    """
    x_adv = x.clone().detach().requires_grad_(True)
    logits = model(x_adv)

    # Loss for the TARGET class — we want to minimize this
    loss = F.cross_entropy(logits, y_target)
    model.zero_grad()
    loss.backward()

    # SUBTRACT: move toward lower loss for target (not higher loss for true label)
    grad_sign = x_adv.grad.sign()
    x_adv = x_adv.detach() - epsilon * grad_sign
    x_adv = torch.clamp(x_adv, x.min().item(), x.max().item())
    return x_adv


# Find a digit 7 and try to make it look like an 8 to the model
model.eval()
for x_b, y_b in test_loader:
    x_b, y_b = x_b.to(device), y_b.to(device)
    # Find a 7
    idx = (y_b == 7).nonzero(as_tuple=True)[0]
    if len(idx) > 0:
        x7 = x_b[idx[:1]]
        y7 = y_b[idx[:1]]
        break

target_class = torch.tensor([8], device=device)  # want model to predict "8"

# Try increasingly large epsilon for targeted attack
print(f"True label: 7 → target: 8")
print(f"{'epsilon':>8} {'clean pred':>12} {'adv pred':>10} {'conf (8)':>10} {'success':>8}")
print("-" * 55)

with torch.no_grad():
    clean_pred = model(x7).argmax(dim=1).item()

for eps in [0.1, 0.2, 0.3, 0.4, 0.5]:
    x_adv = fgsm_targeted(model, x7, target_class, eps)
    with torch.no_grad():
        logits_adv = model(x_adv)
        pred_adv = logits_adv.argmax(dim=1).item()
        conf_8 = F.softmax(logits_adv, dim=1)[0, 8].item()
    success = "✓" if pred_adv == 8 else "✗"
    print(f"{eps:>8.2f} {clean_pred:>12} {pred_adv:>10} {conf_8:>10.3f} {success:>8}")

## 5. Transferability

One of the most important and counterintuitive properties of adversarial examples: they transfer across models. An adversarial example crafted against model A often fools model B — even if B has a completely different architecture and was trained independently.

This was the discovery that elevated adversarial examples from a curiosity to a real security threat. It means:
1. An attacker doesn't need access to the target model — they can attack a substitute model and transfer the examples
2. Defenses that claim "security through obscurity" (keeping weights secret) are weaker than they appear
3. The vulnerability is a property of the loss landscape and data distribution, not a quirk of any single model

Why does it transfer? Because different models trained on the same data learn similar decision boundaries. The adversarial direction found by one model is likely to cross the decision boundary of another model trained on the same distribution.

In [ ]:
# Train a second model with a different architecture and random seed
class MnistMLP(nn.Module):
    """Simple MLP — completely different architecture from the CNN."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(99)  # different seed
model_b = MnistMLP().to(device)
opt_b = torch.optim.Adam(model_b.parameters(), lr=1e-3)

print("Training model B (MLP, different architecture, different seed)...")
for epoch in range(5):
    model_b.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt_b.zero_grad()
        F.cross_entropy(model_b(x), y).backward()
        opt_b.step()
acc_b = evaluate(model_b, test_loader, device)
print(f"Model B clean accuracy: {acc_b:.4f}")

In [ ]:
# Transferability experiment:
# Generate adversarial examples against model A (CNN)
# Evaluate how many of them fool model B (MLP)

model.eval()
model_b.eval()

print("Transferability: adversarial examples crafted against Model A (CNN)")
print("Evaluated on both Model A and Model B (MLP, different architecture)")
print()
print(f"{'ε':>6} {'Model A (source)':>18} {'Model B (transfer)':>20} {'Transfer rate':>15}")
print("-" * 65)

for eps in [0.1, 0.2, 0.3, 0.4]:
    correct_a = correct_b = total = 0
    for i, (x, y) in enumerate(test_loader):
        if i >= 15:
            break
        x, y = x.to(device), y.to(device)
        x_adv = fgsm_attack(model, x, y, eps)  # craft against model A

        with torch.no_grad():
            preds_a = model(x_adv).argmax(1)
            preds_b = model_b(x_adv).argmax(1)

        correct_a += (preds_a == y).sum().item()
        correct_b += (preds_b == y).sum().item()
        total += y.size(0)

    acc_a_adv = correct_a / total  # model A accuracy on adversarial examples
    acc_b_adv = correct_b / total  # model B accuracy on transferred examples
    transfer_rate = 1 - acc_b_adv  # fraction where transfer fooled model B
    print(f"{eps:>6.2f} {acc_a_adv:>16.3f}   {acc_b_adv:>18.3f}   {transfer_rate:>13.3f}")

print()
print("Transfer rate: fraction of adversarial examples that also fool Model B")
print("Even examples crafted against CNN transfer to a fully different MLP architecture")

## 6. Defenses: What Works, What Doesn't

A large fraction of proposed defenses against adversarial examples have been broken. Athalye, Carlini & Wagner (2018) — *"Obfuscated Gradients Give a False Sense of Security"* — broke 7 of 9 papers accepted at ICLR 2018 that proposed defenses. The pattern: defenses that obscure gradient information (preprocessing, stochastic layers, detection networks) can be bypassed by adaptive attacks that work around the obscuration.

**Security theater (usually broken):**
- Input preprocessing (JPEG compression, median filtering, feature squeezing): effective against FGSM but broken by adaptive attacks that account for the preprocessing in the gradient computation
- Detection-based defenses ("is this input adversarial?"): detection can be evaded by minimizing detectability alongside misclassification
- Certified detection: can certify detection for small ε, but the detector itself becomes a target

**What actually works:**
- **Adversarial training** (Madry et al., 2018): train on adversarial examples. Still the most reliable defense. Expensive — requires generating adversarial examples per batch during training. Doesn't fully solve the problem but significantly raises the bar.
- **Certified defenses** (randomized smoothing, interval bound propagation): provide provable guarantees at the cost of accuracy on clean inputs. The only way to make provable claims.
- **Input transformations that are differentiable**: if your defense is differentiable, you can account for it in the attack. Only non-differentiable, non-approximable defenses provide gradient obfuscation that doesn't just delay the attacker.

In [ ]:
# Simple adversarial training: train on a mix of clean and FGSM examples
# This is a simplified version — full adversarial training uses PGD (notebook 03)

torch.manual_seed(42)
model_adv = MnistCNN().to(device)
opt_adv = torch.optim.Adam(model_adv.parameters(), lr=1e-3)

TRAIN_EPSILON = 0.3  # attack strength during training
ADV_FRACTION = 0.5   # fraction of each batch that is adversarial

print(f"Adversarial training with ε={TRAIN_EPSILON}, {ADV_FRACTION:.0%} adversarial examples...")

for epoch in range(5):
    model_adv.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        # Generate adversarial examples for half the batch
        n_adv = int(len(x) * ADV_FRACTION)
        x_adv = fgsm_attack(model_adv, x[:n_adv], y[:n_adv], TRAIN_EPSILON)

        # Mix clean and adversarial examples
        x_mixed = torch.cat([x_adv, x[n_adv:]], dim=0)
        y_mixed = y  # labels unchanged

        opt_adv.zero_grad()
        F.cross_entropy(model_adv(x_mixed), y_mixed).backward()
        opt_adv.step()

    clean_acc = evaluate(model_adv, test_loader, device)
    adv_acc   = evaluate_under_fgsm(model_adv, test_loader, TRAIN_EPSILON, device)
    print(f"  Epoch {epoch+1}/5 — clean: {clean_acc:.4f}  adversarial (ε={TRAIN_EPSILON}): {adv_acc:.4f}")

In [ ]:
# Compare clean model vs adversarially trained model
print("Clean model vs Adversarially trained model")
print(f"{'ε':>6} {'Clean model':>14} {'Adv-trained model':>19} {'Improvement':>13}")
print("-" * 58)

for eps in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    if eps == 0:
        acc_clean = evaluate(model, test_loader, device)
        acc_adv_model = evaluate(model_adv, test_loader, device)
    else:
        acc_clean = evaluate_under_fgsm(model, test_loader, eps, device)
        acc_adv_model = evaluate_under_fgsm(model_adv, test_loader, eps, device)
    improvement = acc_adv_model - acc_clean
    print(f"{eps:>6.2f} {acc_clean:>14.4f} {acc_adv_model:>19.4f} {improvement:>+13.4f}")

print()
print("Note: adversarial training trades some clean accuracy for robustness.")
print("This accuracy-robustness tradeoff is fundamental — Tsipras et al. (2019)")
print("proved it is not just an artifact of the training procedure.")

## 7. Why Gradient Masking Doesn't Work

A common failed defense: make the gradients look unhelpful so the attacker can't compute a useful perturbation. This is called **gradient masking** or **obfuscated gradients**. It takes several forms:

- **Preprocessing**: apply JPEG compression or Gaussian blur before the model sees the input. The preprocessing function may have zero or uninformative gradients.
- **Stochastic components**: add dropout at test time so gradients are noisy.
- **Shattered gradients**: use non-differentiable operations (argmax, rounding) in the forward pass.

These all fail because attackers can use **Expectation over Transformations (EOT)**: average the gradient over many stochastic realizations, or use finite-difference approximations that don't require differentiating through the defense. If you can query the model, you can estimate the gradient.

The attack below shows how to bypass preprocessing-based defenses by accounting for the preprocessing in the gradient computation.

In [ ]:
def gaussian_blur_defense(x: torch.Tensor, sigma: float = 1.0) -> torch.Tensor:
    """Naive defense: blur the input before passing to model."""
    # Simple box blur approximation using avg_pool
    kernel_size = 3
    blurred = F.avg_pool2d(x, kernel_size=kernel_size, stride=1, padding=kernel_size//2)
    return blurred


def evaluate_with_defense(model, loader, defense_fn, device, n_batches=15):
    """Evaluate model accuracy when defense is applied to inputs."""
    model.eval()
    correct = total = 0
    for i, (x, y) in enumerate(loader):
        if i >= n_batches:
            break
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            preds = model(defense_fn(x)).argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / total


def fgsm_bypass_defense(
    model: nn.Module,
    defense_fn,
    x: torch.Tensor,
    y: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    """
    FGSM that accounts for the defense in the gradient computation.
    The attacker computes gradients THROUGH the defense function.
    Works when defense_fn is differentiable (most preprocessing is).
    """
    x_adv = x.clone().detach().requires_grad_(True)
    # Gradient flows THROUGH defense_fn — the defense is not hidden from the attacker
    logits = model(defense_fn(x_adv))
    loss = F.cross_entropy(logits, y)
    model.zero_grad()
    loss.backward()
    grad_sign = x_adv.grad.sign()
    x_adv = x_adv.detach() + epsilon * grad_sign
    x_adv = torch.clamp(x_adv, x.min().item(), x.max().item())
    return x_adv


# Experiment: does blur defense help against naive FGSM? What about adaptive attack?
model.eval()
eps = 0.3

clean_acc = evaluate(model, test_loader, device)
defended_clean_acc = evaluate_with_defense(model, test_loader, gaussian_blur_defense, device)

# Naive FGSM, then blur applied
naive_adv_after_blur = 0
adaptive_adv = 0
total = 0
for i, (x, y) in enumerate(test_loader):
    if i >= 15:
        break
    x, y = x.to(device), y.to(device)

    # Naive: craft FGSM ignoring defense, then apply defense
    x_adv_naive = fgsm_attack(model, x, y, eps)
    with torch.no_grad():
        preds_naive = model(gaussian_blur_defense(x_adv_naive)).argmax(1)
    naive_adv_after_blur += (preds_naive == y).sum().item()

    # Adaptive: craft FGSM that accounts for the defense
    x_adv_adaptive = fgsm_bypass_defense(model, gaussian_blur_defense, x, y, eps)
    with torch.no_grad():
        preds_adaptive = model(gaussian_blur_defense(x_adv_adaptive)).argmax(1)
    adaptive_adv += (preds_adaptive == y).sum().item()

    total += y.size(0)

print("Gradient masking demo: Gaussian blur defense")
print(f"  Clean accuracy (no defense):          {clean_acc:.4f}")
print(f"  Clean accuracy (with blur defense):   {defended_clean_acc:.4f}  ← defense costs some accuracy")
print(f"  Naive FGSM accuracy (with blur):      {naive_adv_after_blur/total:.4f}  ← blur seems to help!")
print(f"  Adaptive FGSM accuracy (with blur):   {adaptive_adv/total:.4f}  ← defense is bypassed")
print()
print("Lesson: any differentiable defense can be bypassed by an adaptive attacker.")
print("'Security through obscurity' fails when the attacker knows the defense exists.")

## 8. Practical Implications for Deployed Systems

**When adversarial examples are a real concern:**
- Your model processes untrusted inputs that an adversary can craft (computer vision APIs, content moderation, biometrics)
- You're operating in a safety-critical domain (medical imaging, autonomous systems, fraud detection)
- Your model output directly controls a high-value decision without human review

**When adversarial examples are not your primary concern:**
- Your input domain is natural language (adversarial examples for text look like normal text, limiting perception — though LLM jailbreaks are a related concept covered in Series 4)
- Your model is one component in a human-reviewed pipeline
- The attacker doesn't have repeated query access to craft perturbations

**Practical mitigations (ordered by reliability):**
1. **Adversarial training** — the most battle-tested. Use PGD-based training (notebook 03), not just FGSM
2. **Randomized smoothing** — provable guarantees at the cost of some clean accuracy
3. **Monitoring and rate limiting** — adversarial example crafting requires many queries; detecting and rate-limiting query bursts slows black-box attacks
4. **Ensemble methods** — harder to transfer to all models simultaneously (but not impossible)
5. **Input validation** — domain-specific checks that reject implausible inputs (won't stop all attacks but raises the bar)

In [ ]:
# Summary visualization: attack success rates and defense effectiveness
epsilons_plot = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

print("Computing final comparison across epsilon values...")
results = {'clean': [], 'adv_trained': []}
for eps in epsilons_plot:
    if eps == 0:
        results['clean'].append(evaluate(model, test_loader, device))
        results['adv_trained'].append(evaluate(model_adv, test_loader, device))
    else:
        results['clean'].append(evaluate_under_fgsm(model, test_loader, eps, device))
        results['adv_trained'].append(evaluate_under_fgsm(model_adv, test_loader, eps, device))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epsilons_plot, results['clean'], 'o-', color='steelblue', linewidth=2, label='Standard training', markersize=6)
ax.plot(epsilons_plot, results['adv_trained'], 's-', color='darkorange', linewidth=2, label='Adversarial training (FGSM)', markersize=6)
ax.axhline(0.1, color='gray', linestyle=':', alpha=0.6, label='Random chance (10 classes)')
ax.fill_between(epsilons_plot, results['clean'], results['adv_trained'],
                alpha=0.15, color='green', label='Robustness gain')

ax.set_xlabel('Epsilon (L∞ perturbation budget)')
ax.set_ylabel('Accuracy under FGSM')
ax.set_title('FGSM Robustness: Standard vs Adversarially Trained Model')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("robustness_comparison.png", bbox_inches='tight')
plt.show()

## Summary

FGSM in one line:
```python
x_adv = x + epsilon * sign(∇_x L(f(x), y))
```

**Key facts:**
- The sign of the gradient — not its magnitude — is what matters under L∞ constraint
- A single gradient step is enough to cross the decision boundary for most naturally-trained models
- Adversarial examples transfer across architectures — the vulnerability is in the data distribution, not the model
- Gradient masking is security theater — any differentiable defense can be bypassed by an adaptive attacker
- Adversarial training raises the bar significantly but comes with a clean-accuracy cost
- Targeted attack: subtract (minimize loss for target class); untargeted: add (maximize loss for true class)

**Next:** Notebook 03 covers PGD (Projected Gradient Descent) — iterated FGSM with projection. PGD is the current standard adversarial attack benchmark and the basis of the most reliable adversarial training procedure (Madry et al., 2018). Understanding PGD is prerequisite to the C&W attack and certified defenses.